In [ ]:
import sys
from pathlib import Path

def find_gptcast_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'gptcast').is_dir():
            return p
        if (p / 'GPTCast' / 'gptcast').is_dir():
            return (p / 'GPTCast').resolve()
        p = p.parent
    return start.resolve()

ROOT = find_gptcast_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Using ROOT:', ROOT)

from gptcast.models import VAEGANVQ
from gptcast.utils.plotting import plot_mutiple_era5land as plot_mutiple
from gptcast.utils.converters import swvl1_norm_to_phys
import numpy as np
import random
from datetime import datetime
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from einops import rearrange

# Reproducibility: crops / random ops can introduce nondeterminism.
try:
    from lightning.pytorch import seed_everything
    seed_everything(42, workers=True)
except Exception:
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


## Load pretrained model and a soil moisture (swvl1) sequence of data from the dataset


In [ ]:
# ROOT is set in the first cell (auto-detected)

# Compare two first-stage checkpoints (update paths as needed)
#
# Upstream GPTCast notebook semantics: `old=vae_mae` and `new=vae_mwae` (downloaded from Zenodo).
# In this SWVL1 fork we do not download upstream precipitation models.
# Here, `old/new` refer to two SWVL1 first-stage checkpoints you choose.
# They can come from different run folders (e.g., if training was interrupted and resumed).

# Prefilled to two ckpts that exist in this repo (as of 2026-03-03):
# - old: earlier run (epoch 31)
# - new: later run (epoch 38)
ckpt_old = ROOT / 'logs/train/runs/2026-03-01_16-06-29/checkpoints/epoch_031.ckpt'
ckpt_new = ROOT / 'logs/train/runs/2026-03-01_18-29-57/checkpoints/epoch_038.ckpt'

assert ckpt_old.exists(), f'Missing: {ckpt_old}'
assert ckpt_new.exists(), f'Missing: {ckpt_new}'

ae_old = VAEGANVQ.load_from_pretrained(str(ckpt_old), device=device).to(device).eval()
ae_new = VAEGANVQ.load_from_pretrained(str(ckpt_new), device=device).to(device).eval()


In [ ]:
# ROOT is set in the first cell (auto-detected)

# This notebook mirrors the original GPTCast AE reconstruction example, but for ERA5-Land SWVL1.
#
# For soil moisture, purely random crops can land on mostly-ocean patches (NaNs). That makes figures
# uninformative. Here we pick a reproducible example centred over East China, and we apply a
# *smart* crop (small jitter search) to reduce ocean fraction while staying near the ROI.

import xarray as xr
from datetime import timedelta
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch.nn.functional as F

# Example start time (daily at 11:00 in this dataset)
T0 = datetime(2020, 4, 6, 11, 0)
SEQ_LEN = 7

# Data location (yearly NetCDF)
BASE_DIR = ROOT / 'data/0.1/1/land_surface'
NC_NAME = 'volumetric_soil_water_layer_1.nc'

# Training-style preprocessing
CLIP = (0.0, 0.8)
RESIZE = 720
CROP = 256

# East China center (degrees)
ROI_NAME = 'East China'
ROI_CENTER = dict(lat=34.0, lon=116.0)

# Smart-crop around the ROI center (on resized grid) to avoid too much ocean
MAX_MASK_FRAC = 0.40
SMART_ATTEMPTS = 50
JITTER_PX = 80


def open_year_ds(year: int) -> xr.Dataset:
    p = BASE_DIR / str(year) / NC_NAME
    assert p.exists(), f"Missing: {p}"
    return xr.open_dataset(p)


def sel_time_swvl1(ds: xr.Dataset, dt: datetime):
    t = np.datetime64(dt)
    try:
        return ds['swvl1'].sel(time=t)
    except Exception:
        # Keep it robust to small timestamp mismatches.
        return ds['swvl1'].sel(time=t, method='nearest')


# Load frames (physical units, NaN over ocean)
frames = []
ds_y = open_year_ds(T0.year)
for j in range(SEQ_LEN):
    dt_j = T0 + timedelta(days=j)
    da = sel_time_swvl1(ds_y, dt_j)
    frames.append(da.values.astype(np.float32, copy=False))
frames = np.stack(frames, axis=0)  # (S, lat, lon)

lat_vals = ds_y['latitude'].values
lon_vals = ds_y['longitude'].values

mask_native = np.isnan(frames[0])

cmin, cmax = float(CLIP[0]), float(CLIP[1])
frames = np.nan_to_num(frames, nan=cmin)
frames = np.clip(frames, cmin, cmax)

# Resize to a square canvas (paper-style)
x = torch.from_numpy(frames).unsqueeze(1)  # (S,1,H,W)
x = F.interpolate(x, size=(RESIZE, RESIZE), mode='bilinear', align_corners=False)
frames_r = x.squeeze(1).numpy()

m = torch.from_numpy(mask_native.astype(np.float32))[None, None, ...]
m = F.interpolate(m, size=(RESIZE, RESIZE), mode='nearest')
mask_r = m[0, 0].numpy().astype(bool)

# Normalize to [-1,1] (same mapping as training)
nmin, nmax = -1.0, 1.0
frames_n = (frames_r - cmin) / (cmax - cmin + 1e-12)
frames_n = frames_n * (nmax - nmin) + nmin

# Map ROI center (lat/lon) to resized pixel indices
lat_c = float(ROI_CENTER['lat'])
lon_c = float(ROI_CENTER['lon'])

lat_idx = int(np.argmin(np.abs(lat_vals - lat_c)))
lon_idx = int(np.argmin(np.abs(lon_vals - lon_c)))

y_center = int(round(lat_idx / (len(lat_vals) - 1) * (RESIZE - 1)))
x_center = int(round(lon_idx / (len(lon_vals) - 1) * (RESIZE - 1)))


def clamp(v, lo, hi):
    return int(max(lo, min(hi, v)))


best = None
best_frac = 1.0

for k in range(SMART_ATTEMPTS):
    dy = random.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    dx = random.randint(-JITTER_PX, JITTER_PX) if k > 0 else 0
    y0 = clamp(y_center - CROP // 2 + dy, 0, RESIZE - CROP)
    x0 = clamp(x_center - CROP // 2 + dx, 0, RESIZE - CROP)

    frac = float(mask_r[y0 : y0 + CROP, x0 : x0 + CROP].mean())
    if frac < best_frac:
        best_frac = frac
        best = (y0, x0)
    if frac <= MAX_MASK_FRAC:
        best = (y0, x0)
        best_frac = frac
        break

y0, x0 = best
print(f"ROI {ROI_NAME} center(lat={lat_c}, lon={lon_c}) -> resized center(y={y_center}, x={x_center})")
print(f"Selected crop top-left(y0={y0}, x0={x0}), mask_frac={best_frac:.3f}")

# Global context plot (first frame) + crop box
# Note: this is the resized field (same canvas used for patch extraction).
global0 = np.ma.array(frames_r[0], mask=mask_r)
cm = plt.get_cmap('viridis').copy()
cm.set_bad(color='lightgray')

fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=160, layout='constrained')
ax.imshow(global0, cmap=cm, vmin=cmin, vmax=cmax, interpolation='nearest')
ax.add_patch(Rectangle((x0, y0), CROP, CROP, linewidth=1.0, edgecolor='red', facecolor='none'))
ax.set_title(f"Global swvl1 (resized) {T0}  ROI: {ROI_NAME}")
ax.axis('off')
plt.show()

# Build an 'el' dict like the dataloader output (batch_size=1) for downstream cells.
patch = frames_n[:, y0 : y0 + CROP, x0 : x0 + CROP]  # (S,H,W) in [-1,1]
mask_patch = mask_r[y0 : y0 + CROP, x0 : x0 + CROP]

image_hws = np.transpose(patch, (1, 2, 0))  # (H,W,S)
el = {
    'image': torch.from_numpy(image_hws.astype(np.float32, copy=False))[None, ...],
    'mask': torch.from_numpy(mask_patch)[None, ...],
    'file_path_': [T0.strftime('SM_%Y%m%d%H%M')],
}


In [ ]:
dt = datetime.strptime(el['file_path_'][0], 'SM_%Y%m%d%H%M')
mask = el['mask'].cpu().numpy().squeeze()
data = rearrange(el['image'].cpu().numpy().squeeze(), 'h w s -> s h w')
input_data = np.ma.array(data, mask=np.broadcast_to(mask, data.shape))


## Reconstruct the input sequence with both autencoders and compare the results


In [ ]:
def reconstruct_swvl1(ae: VAEGANVQ, arr: np.ma.MaskedArray):
    assert arr.ndim == 3
    x = arr.data.astype(np.float32, copy=False)
    # Treat time as batch dimension (S,1,H,W)
    x = torch.tensor(x, dtype=torch.float32, device=device)[:, None, ...]
    with torch.no_grad():
        y, _ = ae(x, auto_pad=True)
    y = y.detach().cpu().numpy().squeeze().clip(-1, 1)
    return np.ma.array(y, mask=arr.mask)

new_rec = reconstruct_swvl1(ae_new, input_data)
old_rec = reconstruct_swvl1(ae_old, input_data)


### plot input, new and old autoencoder output


In [ ]:
plot_mutiple(swvl1_norm_to_phys(input_data), title=f"Input {dt}", figsize=(10,3))
plot_mutiple(swvl1_norm_to_phys(new_rec), title=f"Reconstructed (ckpt new)", figsize=(10,3))
plot_mutiple(swvl1_norm_to_phys(old_rec), title=f"Reconstructed (ckpt old)", figsize=(10,3))

# Quantitative check (masked MAE/RMSE in physical units m3/m3)
def per_lead_mae_rmse(pred: np.ma.MaskedArray, tgt: np.ma.MaskedArray):
    diff = pred - tgt
    mae = np.ma.mean(np.abs(diff), axis=(1, 2))
    rmse = np.sqrt(np.ma.mean(diff ** 2, axis=(1, 2)))
    return mae, rmse

inp_phys = swvl1_norm_to_phys(input_data)
new_phys = swvl1_norm_to_phys(new_rec)
old_phys = swvl1_norm_to_phys(old_rec)

mae_n, rmse_n = per_lead_mae_rmse(new_phys, inp_phys)
mae_o, rmse_o = per_lead_mae_rmse(old_phys, inp_phys)

print(f"Reconstruction metrics over {inp_phys.shape[0]} frames (masked/ocean ignored):")
print(f"  new_ckpt: mean MAE={float(mae_n.mean()):.4f}, mean RMSE={float(rmse_n.mean()):.4f}")
print(f"  old_ckpt: mean MAE={float(mae_o.mean()):.4f}, mean RMSE={float(rmse_o.mean()):.4f}")
